# Ex.4: Efficient inference: quantization, KV cache, speculative decoding
**EECE.4860/5860**: ungraded practice for Lectures 7–8. **Needs a CUDA GPU** (Colab T4); about 8 minutes.

We measure, on Qwen3-1.7B: (1) memory, speed and perplexity in fp16, 8-bit and 4-bit (bitsandbytes); (2) decoding with and without the KV cache; (3) assisted (speculative) decoding with Qwen3-0.6B as the draft model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/exercises/ex4_efficient_inference.ipynb)

In [ ]:
import importlib.util
if importlib.util.find_spec("genai_utils") is None:
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')
if importlib.util.find_spec("bitsandbytes") is None:
    get_ipython().run_line_magic("pip", "install -q bitsandbytes")
import gc, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from genai_utils import setup_env, measure
device, dtype = setup_env()
assert device == "cuda", "This exercise needs a CUDA GPU (Runtime > Change runtime type > T4 GPU)."
BIG, SMALL = "Qwen/Qwen3-1.7B", "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(BIG)
TEXT = ("Generative models learn the probability distribution of their training data and can sample new examples from it. "
        "Large language models are trained to predict the next token, and at inference time they generate text one token at a time.")
prompt = tok.apply_chat_template([{"role": "user", "content": "Explain in one paragraph how a refrigerator works."}],
                                 tokenize=False, add_generation_prompt=True, enable_thinking=False)
enc = tok(prompt, return_tensors="pt").to(device)

def perplexity(model):
    ids = tok(TEXT, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        return torch.exp(model(ids, labels=ids).loss).item()

def tokens_per_second(model, n=128, **kw):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model.generate(**enc, max_new_tokens=n, min_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id, **kw)
    torch.cuda.synchronize()
    return n / (time.perf_counter() - t0), out

## 1. Quantization: fp16 vs. 8-bit vs. 4-bit (NF4)

In [ ]:
configs = {"fp16": None,
           "int8 (LLM.int8)": BitsAndBytesConfig(load_in_8bit=True),
           "nf4 (4-bit)": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)}
rows = []
for name, q in configs.items():
    m = AutoModelForCausalLM.from_pretrained(BIG, dtype=torch.float16, quantization_config=q, device_map={"": 0})
    tps, _ = tokens_per_second(m)
    rows.append((name, m.get_memory_footprint() / 2**30, tps, perplexity(m)))
    del m; gc.collect(); torch.cuda.empty_cache()
print(f"{'format':18s}{'weights GiB':>12s}{'tokens/s':>10s}{'perplexity':>12s}")
for r in rows:
    print(f"{r[0]:18s}{r[1]:12.2f}{r[2]:10.1f}{r[3]:12.2f}")

## 2. The KV cache
Without the cache, every new token recomputes attention keys and values for the whole prefix.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(BIG, dtype=torch.float16).to(device)
with_cache, out1 = tokens_per_second(model, use_cache=True)
without_cache, out2 = tokens_per_second(model, use_cache=False)
print(f"with KV cache: {with_cache:.1f} tok/s, without: {without_cache:.1f} tok/s; identical output: {torch.equal(out1, out2)}")

## 3. Assisted (speculative) decoding
A small draft model proposes several tokens; the large model checks them in one forward pass. With greedy decoding the output is identical to the large model alone (Leviathan et al., 2023).

In [ ]:
draft = AutoModelForCausalLM.from_pretrained(SMALL, dtype=torch.float16).to(device)
plain, base_out = tokens_per_second(model)
assisted, asst_out = tokens_per_second(model, assistant_model=draft)
print(f"plain: {plain:.1f} tok/s, assisted: {assisted:.1f} tok/s; identical output: {torch.equal(base_out, asst_out)}")
print(tok.decode(asst_out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)[:400])

## 4. Think about it
* Which format would you deploy on a 16 GB GPU serving many users, and why?
* Why does the speed-up from assisted decoding depend on how often the draft model's tokens are accepted?